# Zero-DCE — Baseline Reproduction

Reproduces the original Zero-DCE result and records the baseline numbers used
throughout this project: full-reference metrics on LOL eval15, parameter count,
FLOPs, runtime, and the curve-parameter maps.

**Repository branch:** `master`
**Checkpoint produced:** `Epoch199.pth` (saved to Drive)

Run cells top to bottom. Cell 2 (training) takes hours — skip it if
`Epoch199.pth` is already on your Drive.

> Cleaned from `ZERODCE_1.ipynb`. All repeated `git pull` cells were collapsed
> into the single setup cell below.

## 1. Setup — repository, Drive, GPU, dependencies

The **only** git pull in this notebook.

In [ ]:
# ============================================================
# SETUP — single repository pull, Drive mount, dependencies
# ============================================================

import os, sys, subprocess

REPO_DIR = "/content/Zero-DCE"
BRANCH   = "master"
CODE_DIR = f"{REPO_DIR}/Zero-DCE_code"

if not os.path.isdir(REPO_DIR):
    subprocess.run(
        ["git", "clone", "https://github.com/shreyaio/Zero-DCE.git", REPO_DIR],
        check=True,
    )

def git_run(args):
    return subprocess.check_output(["git"] + args, cwd=REPO_DIR, text=True).strip()

git_run(["fetch", "origin", BRANCH])
git_run(["checkout", BRANCH])
git_run(["pull", "--ff-only", "origin", BRANCH])

print("Branch:", git_run(["branch", "--show-current"]))
print("Commit:", git_run(["rev-parse", "HEAD"]))
print(git_run(["log", "--oneline", "-3"]))

os.chdir(CODE_DIR)
if CODE_DIR not in sys.path:
    sys.path.insert(0, CODE_DIR)

# ---- Drive ----
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

DRIVE_ROOT = "/content/drive/MyDrive/zero-dce"
for sub in ["checkpoints", "results", "data"]:
    os.makedirs(f"{DRIVE_ROOT}/{sub}", exist_ok=True)
print("Drive ready at", DRIVE_ROOT)

# ---- GPU ----
import torch
print("torch:", torch.__version__, "| CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("NO GPU — Runtime > Change runtime type > GPU")

# ---- dependencies ----
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "thop", "scikit-image"], check=False)

# ---- source modules import cleanly? ----
import model, dataloader, Myloss
net_probe = model.enhance_net_nopool()
print("Modules OK. DCE-Net params:", sum(p.numel() for p in net_probe.parameters()))

## 2. Training data (SICE Part1) — download and extract

In [ ]:
# ============================================================
# TRAINING DATA — cached on Drive, extracted locally
# ============================================================

import os, subprocess, sys

DRIVE_DATA = "/content/drive/MyDrive/zero-dce/data"
ZIP_PATH   = f"{DRIVE_DATA}/train_data.zip"
FILE_ID    = "1GAB3uGsmAyLgtDBDONbil08vVu5wJcG3"

os.makedirs(DRIVE_DATA, exist_ok=True)

if not os.path.exists(ZIP_PATH):
    print("Downloading training data to Drive (one-time)...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gdown"], check=False)
    subprocess.run(["gdown", "--id", FILE_ID, "-O", ZIP_PATH], check=True)
else:
    print("Using cached zip:", ZIP_PATH)

print("Zip size (MB):", round(os.path.getsize(ZIP_PATH) / 1e6, 1))

LOCAL_DATA_DIR = "/content/Zero-DCE/Zero-DCE_code/data"
os.makedirs(LOCAL_DATA_DIR, exist_ok=True)
subprocess.run(["unzip", "-q", "-o", ZIP_PATH, "-d", LOCAL_DATA_DIR], check=True)

n_jpg = subprocess.check_output(
    f'find {LOCAL_DATA_DIR}/train_data -name "*.jpg" | wc -l',
    shell=True, text=True).strip()
print("Training images found:", n_jpg)

## 3. Sanity check — bundled pretrained `Epoch99.pth`

Confirms the released weights run before we train our own.

In [ ]:
import sys, os
sys.path.insert(0, "/content/Zero-DCE/Zero-DCE_code")
os.chdir("/content/Zero-DCE/Zero-DCE_code")

import torch, numpy as np
from PIL import Image
import matplotlib.pyplot as plt
import model

assert os.path.exists("snapshots/Epoch99.pth"), "pretrained checkpoint missing from clone"

DCE_net = model.enhance_net_nopool().cuda()
DCE_net.load_state_dict(torch.load("snapshots/Epoch99.pth", map_location="cuda"))
DCE_net.eval()

def enhance(image_path):
    data_lowlight = Image.open(image_path).convert("RGB")
    data_lowlight = np.asarray(data_lowlight) / 255.0
    data_lowlight = torch.from_numpy(data_lowlight).float().permute(2, 0, 1).cuda().unsqueeze(0)
    with torch.no_grad():
        _, enhanced, _ = DCE_net(data_lowlight)
    out = enhanced.squeeze(0).permute(1, 2, 0).cpu().numpy()
    return np.clip(out, 0, 1)

sample_paths = [
    "data/test_data/DICM/01.jpg",
    "data/test_data/DICM/10.jpg",
    "data/test_data/LIME/1.bmp",
    "data/test_data/LIME/5.bmp",
]

DRIVE_RESULTS = "/content/drive/MyDrive/zero-dce/results/sanity_check"
os.makedirs(DRIVE_RESULTS, exist_ok=True)

fig, axes = plt.subplots(len(sample_paths), 2, figsize=(8, 4 * len(sample_paths)))
for i, p in enumerate(sample_paths):
    inp = np.asarray(Image.open(p).convert("RGB")) / 255.0
    out = enhance(p)
    axes[i, 0].imshow(inp); axes[i, 0].set_title(f"input: {os.path.basename(p)}"); axes[i, 0].axis("off")
    axes[i, 1].imshow(out); axes[i, 1].set_title("Zero-DCE (Epoch99.pth)"); axes[i, 1].axis("off")
    Image.fromarray((out * 255).astype(np.uint8)).save(f"{DRIVE_RESULTS}/{os.path.basename(p)}_enhanced.png")

plt.tight_layout()
plt.savefig(f"{DRIVE_RESULTS}/sanity_check_grid.png", dpi=120)
plt.show()
print("Saved to", DRIVE_RESULTS)

## 4. Train the baseline (200 epochs)

**Long-running.** Skip if `/content/drive/MyDrive/zero-dce/checkpoints/Epoch199.pth`
already exists — every later notebook uses that checkpoint.

In [ ]:
import sys, os, glob, re, argparse, importlib
sys.path.insert(0, "/content/Zero-DCE/Zero-DCE_code")
os.chdir("/content/Zero-DCE/Zero-DCE_code")

import lowlight_train
importlib.reload(lowlight_train)

CKPT_DIR = "/content/drive/MyDrive/zero-dce/checkpoints"
os.makedirs(CKPT_DIR, exist_ok=True)
NUM_EPOCHS = 200

existing = glob.glob(f"{CKPT_DIR}/Epoch*.pth")
def epoch_num(p):
    m = re.search(r"Epoch(\d+)\.pth", os.path.basename(p))
    return int(m.group(1)) if m else -1

if existing:
    existing.sort(key=epoch_num)
    latest = existing[-1]
    start_epoch = epoch_num(latest) + 1
    load_pretrain, pretrain_dir = True, latest
    print(f"Resuming from {latest} -> starting at epoch {start_epoch}")
else:
    start_epoch, load_pretrain, pretrain_dir = 0, False, ""
    print("No checkpoint found on Drive — starting fresh from epoch 0")

if start_epoch >= NUM_EPOCHS:
    print(f"Already trained through epoch {start_epoch - 1} (target {NUM_EPOCHS - 1}). Nothing to do.")
else:
    config = argparse.Namespace(
        lowlight_images_path="data/train_data/",
        lr=0.0001,
        weight_decay=0.0001,
        grad_clip_norm=0.1,
        num_epochs=NUM_EPOCHS,
        start_epoch=start_epoch,
        train_batch_size=8,
        val_batch_size=4,
        num_workers=4,
        display_iter=50,      # repo default is 10 — just less print spam, no effect on training
        snapshot_iter=200,    # repo default is 10 (saves ~25x/epoch); 200 saves ~once/epoch, less Drive I/O
        snapshots_folder=CKPT_DIR + "/",
        load_pretrain=load_pretrain,
        pretrain_dir=pretrain_dir,
    )
    import time
    t0 = time.time()
    lowlight_train.train(config)
    print(f"Done. Elapsed: {(time.time() - t0) / 60:.1f} min")

## 5. LOL eval15 dataset

In [ ]:
from huggingface_hub import hf_hub_download
import shutil, os

zip_path = hf_hub_download(repo_id="geekyrakshit/LoL-Dataset", filename="lol_dataset.zip", repo_type="dataset")
print("Downloaded to:", zip_path)

EXTRACT_DIR = "/content/lol_dataset"
os.makedirs(EXTRACT_DIR, exist_ok=True)
shutil.unpack_archive(zip_path, EXTRACT_DIR)

print("--- structure ---")
!find {EXTRACT_DIR} -maxdepth 3 | head -40
print("--- total files ---")
!find {EXTRACT_DIR} -type f | wc -l

## 6. Table 6 substitute — full-reference metrics on LOL eval15

SICE Part2 was unreachable; LOL eval15 used instead. **Not** directly comparable to the paper.

In [ ]:
!pip install -q scikit-image

import os, sys, json, statistics
import numpy as np
import torch
from PIL import Image
from skimage.metrics import peak_signal_noise_ratio as psnr_fn
from skimage.metrics import structural_similarity as ssim_fn

sys.path.insert(0, "/content/Zero-DCE/Zero-DCE_code")
os.chdir("/content/Zero-DCE/Zero-DCE_code")
import model

LOW_DIR = "/content/lol_dataset/lol_dataset/eval15/low"
HIGH_DIR = "/content/lol_dataset/lol_dataset/eval15/high"
low_files = sorted(f for f in os.listdir(LOW_DIR) if not f.startswith("."))
high_files = sorted(f for f in os.listdir(HIGH_DIR) if not f.startswith("."))
assert low_files == high_files, f"filename mismatch: {set(low_files) ^ set(high_files)}"
print(f"{len(low_files)} paired images confirmed, e.g. {low_files[:3]}")

CKPT_DIR = "/content/drive/MyDrive/zero-dce/checkpoints"
ckpt_path = f"{CKPT_DIR}/Epoch199.pth"
assert os.path.exists(ckpt_path), f"missing final checkpoint at {ckpt_path} — check what's actually in {CKPT_DIR}"

DCE_net = model.enhance_net_nopool().cuda()
DCE_net.load_state_dict(torch.load(ckpt_path, map_location="cuda"))
DCE_net.eval()

def enhance(image_path):
    data_lowlight = Image.open(image_path).convert("RGB")
    data_lowlight = np.asarray(data_lowlight) / 255.0
    t = torch.from_numpy(data_lowlight).float().permute(2, 0, 1).cuda().unsqueeze(0)
    with torch.no_grad():
        _, enhanced, _ = DCE_net(t)
    out = enhanced.squeeze(0).permute(1, 2, 0).cpu().numpy()
    return np.clip(out, 0, 1)

DRIVE_RESULTS = "/content/drive/MyDrive/zero-dce/results/lol_eval15"
os.makedirs(DRIVE_RESULTS, exist_ok=True)

records = []
for fname in low_files:
    low_path, high_path = f"{LOW_DIR}/{fname}", f"{HIGH_DIR}/{fname}"
    out = enhance(low_path)
    gt_img = Image.open(high_path).convert("RGB")
    if (gt_img.height, gt_img.width) != out.shape[:2]:
        gt_img = gt_img.resize((out.shape[1], out.shape[0]))
    gt = np.asarray(gt_img).astype(np.float64)
    out_255 = (out * 255).astype(np.float64)

    mae = float(np.mean(np.abs(out_255 - gt)))
    p = float(psnr_fn(gt, out_255, data_range=255))
    s = float(ssim_fn(gt, out_255, data_range=255, channel_axis=2))
    records.append({"file": fname, "psnr": p, "ssim": s, "mae": mae})
    Image.fromarray((out * 255).astype(np.uint8)).save(f"{DRIVE_RESULTS}/{fname}")

avg_psnr = statistics.mean(r["psnr"] for r in records)
avg_ssim = statistics.mean(r["ssim"] for r in records)
avg_mae = statistics.mean(r["mae"] for r in records)
print(f"Zero-DCE on LOL eval15 (n={len(records)}): PSNR {avg_psnr:.2f}  SSIM {avg_ssim:.3f}  MAE {avg_mae:.2f}")

with open(f"{DRIVE_RESULTS}/metrics.json", "w") as f:
    json.dump({"per_image": records, "average": {"psnr": avg_psnr, "ssim": avg_ssim, "mae": avg_mae}}, f, indent=2)
print("Saved to", DRIVE_RESULTS)

## 7. Table 7 — parameters, FLOPs, runtime

In [ ]:
from thop import profile
import torch, time
import model

DCE_net = model.enhance_net_nopool().cuda().eval()
DCE_net.load_state_dict(torch.load("/content/drive/MyDrive/zero-dce/checkpoints/Epoch199.pth", map_location="cuda"))

dummy = torch.randn(1, 3, 1200, 900).cuda()  # paper's eval input size
macs, params = profile(DCE_net, inputs=(dummy,), verbose=False)
print(f"Params: {params:,.0f}")
print(f"thop MACs (for 1200x900x3 input): {macs/1e9:.2f} G")
print(f"  -> as 'FLOPs' = 2x MACs convention: {2*macs/1e9:.2f} G  (paper reports 84.99G for Zero-DCE)")

with torch.no_grad():
    for _ in range(5):
        _ = DCE_net(dummy)
    torch.cuda.synchronize()
    N = 30
    t0 = time.time()
    for _ in range(N):
        _ = DCE_net(dummy)
    torch.cuda.synchronize()
    elapsed = (time.time() - t0) / N
print(f"Own T4 runtime (1200x900x3, avg of {N} runs): {elapsed*1000:.2f} ms")
print("Paper reports 0.0025s on a 2080Ti+i9-10920X — not directly comparable hardware, report side-by-side with that caveat")

## 8. Figures 3 / 6 — curve-parameter maps

In [ ]:
import torch, numpy as np
from PIL import Image
import matplotlib.pyplot as plt
import model

DCE_net = model.enhance_net_nopool().cuda().eval()
DCE_net.load_state_dict(torch.load("/content/drive/MyDrive/zero-dce/checkpoints/Epoch199.pth", map_location="cuda"))

def get_maps(image_path):
    img = Image.open(image_path).convert("RGB")
    data_lowlight = np.asarray(img) / 255.0
    t = torch.from_numpy(data_lowlight).float().permute(2, 0, 1).cuda().unsqueeze(0)
    with torch.no_grad():
        _, enhanced, r = DCE_net(t)
    # r: 1 x 24 x H x W -> 8 iterations x 3 (RGB) channels each
    r = r.squeeze(0).cpu().numpy()
    r_per_iter = r.reshape(8, 3, r.shape[1], r.shape[2])
    avg_map = r_per_iter.mean(axis=0)          # averaged over 8 iterations, per Fig 3
    avg_map_norm = (avg_map - avg_map.min()) / (avg_map.max() - avg_map.min() + 1e-8)
    out = enhanced.squeeze(0).permute(1, 2, 0).cpu().numpy().clip(0, 1)
    return np.asarray(img) / 255.0, avg_map_norm, out

image_path = "data/test_data/DICM/01.jpg"
inp, maps, out = get_maps(image_path)

fig, axes = plt.subplots(1, 5, figsize=(20, 4))
axes[0].imshow(inp); axes[0].set_title("input"); axes[0].axis("off")
for i, ch in enumerate(["R", "G", "B"]):
    im = axes[i+1].imshow(maps[i], cmap="jet", vmin=0, vmax=1)
    axes[i+1].set_title(f"avg curve param A^{ch}"); axes[i+1].axis("off")
axes[4].imshow(out); axes[4].set_title("enhanced"); axes[4].axis("off")
plt.colorbar(im, ax=axes[1:4].tolist(), fraction=0.025)

DRIVE_RESULTS = "/content/drive/MyDrive/zero-dce/results/curve_maps"
import os; os.makedirs(DRIVE_RESULTS, exist_ok=True)
plt.savefig(f"{DRIVE_RESULTS}/curve_param_maps.png", dpi=120, bbox_inches="tight")
plt.show()
print("Saved to", DRIVE_RESULTS)


---
## Recorded baseline

| Metric | This reproduction (T4) | Paper (2080Ti) |
|---|---|---|
| PSNR (LOL eval15) | 17.26 | 16.57 (SICE Part2) |
| SSIM | 0.556 | 0.59 |
| MAE | 32.91 | 98.78 |
| Params | 79,416 | 79,416 / 79,532 |
| FLOPs @1200x900x3 | 85.54 G | 84.99 G |
| Runtime (naive) | 114.04 ms | 2.5 ms |

Runtime is not comparable — the paper uses CUDA-event timing with warm cuDNN.
The paper's own Tables 4 and 7 disagree on parameter count by 116.